In [ ]:
# Cell 1: Imports and settings

from __future__ import annotations
import argparse
import gc
import hashlib
import itertools
import json
import math
import os
import shutil
import sys
import tempfile
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple
import numpy as np
import pandas as pd
from scipy import signal
from scipy.stats import gamma as scipy_gamma
from sklearn.linear_model import LogisticRegression, Ridge

class _EngineProxy:
    def __init__(self, prefix):
        object.__setattr__(self, "_prefix", prefix)
    def __getattr__(self, name):
        key = object.__getattribute__(self, "_prefix") + name
        if key not in globals():
            raise AttributeError(name)
        return globals()[key]
    def __setattr__(self, name, value):
        globals()[object.__getattribute__(self, "_prefix") + name] = value

source_RANDOM_SEED = 20260911

source_GRID_DT = 0.5

source_GRID_END = 20.0

source_BASELINE = (-5.0, 0.0)

source_FULL_T = np.arange(source_BASELINE[0], source_GRID_END + source_GRID_DT / 2, source_GRID_DT)

source_POST_MASK = source_FULL_T >= 0

source_GRID = source_FULL_T[source_POST_MASK]

source_N_TRAIN_POS = 30

source_N_TRAIN_NEG = 15

source_N_TEST_POS = 24

source_N_TEST_NEG = 12

source_FEATURE_ARRAY_KEYS = {'obs', 'state_feat', 'ch_anti', 'clean'}

hb_GRID_DT = 0.5

hb_GRID_END = 20.0

hb_GRID = np.arange(0.0, hb_GRID_END + hb_GRID_DT / 2, hb_GRID_DT)

hb_MIN_GAP = 0.5

hb_MISS_PENALTY_SEC = 4.0

hb_CHANGE_WINDOWS = (3, 5, 9)

hb_RANDOM_SEED = 20260912

hb_PREDICTOR_KEYS = ('obs', 'state_feat')

ALL_PIDS = tuple((f'VP{i:03d}' for i in range(1, 27)))

ACTIVE_RUN_SEED = 1

INNER_FOLDS = 3

BOOT_REPS = 20000

SIGNFLIP_REPS = 20000

BASE_GLOBAL = dict(state_C=0.01, change_frac=0.3, global_alpha=10.0, response_threshold=0.5)

GLOBAL_TIMING_GRID = [dict(BASE_GLOBAL), dict(BASE_GLOBAL, state_C=0.003), dict(BASE_GLOBAL, state_C=0.03), dict(BASE_GLOBAL, change_frac=0.2), dict(BASE_GLOBAL, change_frac=0.4), dict(BASE_GLOBAL, global_alpha=3.0), dict(BASE_GLOBAL, global_alpha=30.0)]

GLOBAL_RESPONSE_GRID = GLOBAL_TIMING_GRID + [dict(BASE_GLOBAL, response_threshold=0.4), dict(BASE_GLOBAL, response_threshold=0.6)]

BASE_LBR = dict(radius=1.0, local_alpha=100.0, strength=0.5)

LBR_GRID = [dict(BASE_LBR), dict(BASE_LBR, radius=0.5), dict(BASE_LBR, radius=1.5), dict(BASE_LBR, local_alpha=30.0), dict(BASE_LBR, local_alpha=300.0), dict(BASE_LBR, strength=0.25), dict(BASE_LBR, strength=0.75)]

COHERENT_SEED = 1

ROW_BASE = 'BASE_CANONICAL_HRF_GLM_DERIVATIVES'

ROW_A = 'BASE_PLUS_A_TEMPORAL_BOUNDARY'

ROW_AB = 'BASE_PLUS_A_PLUS_B_RECOVERY'

ROW_ABC = 'BASE_PLUS_A_PLUS_B_PLUS_C_RESPONSE'

ROW_FULL = 'BASE_PLUS_A_PLUS_B_PLUS_C_PLUS_LBR'

ROW_ORDER = (ROW_BASE, ROW_A, ROW_AB, ROW_ABC, ROW_FULL)

ROW_LABELS = {ROW_BASE: 'Base — Canonical HRF GLM + temporal + dispersion derivatives', ROW_A: 'Base + A', ROW_AB: 'Base + A + B', ROW_ABC: 'Base + A + B + C', ROW_FULL: 'Base + A + B + C + LBR — HemoBoundary'}

ROW_MEANING = {ROW_BASE: 'Conventional complete-output base estimator', ROW_A: 'Add Temporal Boundary Estimation', ROW_AB: 'Add Recovery Estimation', ROW_ABC: 'Add Response Detection', ROW_FULL: 'HemoBoundary with Local Boundary Refinement'}

BASE_RESPONSE_C = 0.5

BASE_RESPONSE_THRESHOLD = 0.5

BASE_BOUNDARY_FRACTION = 0.2

BASE_GLM_RIDGE = 1e-06

LOCAL_OOF_FOLDS_COHERENT = 3

A_SELECTION_GRID = [dict(state_C=0.01, timing_alpha=10.0), dict(state_C=0.003, timing_alpha=10.0), dict(state_C=0.03, timing_alpha=10.0), dict(state_C=0.01, timing_alpha=3.0), dict(state_C=0.01, timing_alpha=30.0)]

B_CHANGE_FRAC_GRID = (0.2, 0.3, 0.4)

C_RESPONSE_THRESHOLD_GRID = (0.4, 0.5, 0.6)

LBR_SELECTION_GRID = [dict(radius=1.0, local_alpha=100.0, strength=0.5), dict(radius=0.5, local_alpha=100.0, strength=0.5), dict(radius=1.5, local_alpha=100.0, strength=0.5), dict(radius=1.0, local_alpha=30.0, strength=0.5), dict(radius=1.0, local_alpha=300.0, strength=0.5), dict(radius=1.0, local_alpha=100.0, strength=0.25), dict(radius=1.0, local_alpha=100.0, strength=0.75)]

ANALYSIS_NAME = 'HemoBoundary main analysis'

MAIN_OUTPUT_DIRNAME = 'analysis/HemoBoundary_Main'

RUN_PART_NAME = 'ALL_SEEDS_01_10'

RUN_SEEDS = tuple(range(1, 11))

EXPECTED_RUN_PART_NAME = 'ALL_SEEDS_01_10'

EXPECTED_RUN_SEEDS = tuple(range(1, 11))


In [ ]:
# Cell 2: Data loading

from __future__ import annotations

def source_participant_files(root: Path) -> Dict[str, Path]:
    files = sorted((root / '02_preprocessed').glob('VP*_WG_ALIGNED_PREPROCESSED.npz'))
    if not files:
        raise FileNotFoundError('No VP*_WG_ALIGNED_PREPROCESSED.npz files found.')
    return {p.name.split('_WG_')[0]: p for p in files}

def source_load_npz(path: Path) -> Dict[str, Any]:
    with np.load(path, allow_pickle=False) as z:
        d = {k: z[k] for k in z.files}
    required = ['nirs_times', 'hbo_causal_lp02_epochs', 'hbr_causal_lp02_epochs']
    missing = [k for k in required if k not in d]
    if missing:
        raise RuntimeError(f'{path.name}: missing {missing}')
    return d

@dataclass
class source_NoiseProfile:
    n_channels: int
    hbo_scale: np.ndarray
    hbr_scale: np.ndarray
    phi_hbo: float
    phi_hbr: float

def source_safe_lag1(a: np.ndarray) -> float:
    vals = []
    for row in a:
        x, y = (row[:-1], row[1:])
        if len(x) >= 3 and np.std(x) > 1e-10 and (np.std(y) > 1e-10):
            vals.append(float(np.corrcoef(x, y)[0, 1]))
    return float(np.clip(np.nanmedian(vals) if vals else 0.5, 0.0, 0.95))

def source_estimate_noise_profile(path: Path) -> source_NoiseProfile:
    d = source_load_npz(path)
    t = np.asarray(d['nirs_times'], float)
    H = np.asarray(d['hbo_causal_lp02_epochs'], float)
    R = np.asarray(d['hbr_causal_lp02_epochs'], float)
    bm = (t >= source_BASELINE[0]) & (t < source_BASELINE[1])
    if bm.sum() < 4:
        raise RuntimeError(f'{path.name}: insufficient pre-cue samples.')
    C = min(H.shape[1], R.shape[1])
    H = H[:, :C, :][:, :, bm]
    R = R[:, :C, :][:, :, bm]
    H = H - np.nanmedian(H, axis=2, keepdims=True)
    R = R - np.nanmedian(R, axis=2, keepdims=True)
    hs = np.nanmedian(np.nanstd(H, axis=2), axis=0)
    rs = np.nanmedian(np.nanstd(R, axis=2), axis=0)
    hs = np.where(np.isfinite(hs) & (hs > 1e-08), hs, np.nanmedian(hs[hs > 0]) if np.any(hs > 0) else 1.0)
    rs = np.where(np.isfinite(rs) & (rs > 1e-08), rs, np.nanmedian(rs[rs > 0]) if np.any(rs > 0) else 1.0)
    sample_H = H[:min(24, len(H))].reshape(-1, H.shape[-1])
    sample_R = R[:min(24, len(R))].reshape(-1, R.shape[-1])
    return source_NoiseProfile(C, hs, rs, source_safe_lag1(sample_H), source_safe_lag1(sample_R))

def source_load_cache(path: Path) -> Dict[str, Dict[str, np.ndarray]]:
    with np.load(path, allow_pickle=False) as z:
        out = {'train': {}, 'test': {}}
        for k in z.files:
            if k.startswith('tr_'):
                split = 'train'
                key = k[3:]
            elif k.startswith('te_'):
                split = 'test'
                key = k[3:]
            else:
                continue
            arr = z[k]
            if key in source_FEATURE_ARRAY_KEYS:
                arr = source_finite_model_array(arr)
            out[split][key] = arr
    source_assert_finite_model_inputs(out['train'], f'{path.name}:train')
    source_assert_finite_model_inputs(out['test'], f'{path.name}:test')
    return out

def source_prepare_all_caches(file_map: Mapping[str, Path], cache_dir: Path, mode: str) -> Dict[str, Dict[str, Dict[str, np.ndarray]]]:
    common_c = min((source_load_npz(p)['hbo_causal_lp02_epochs'].shape[1] for p in file_map.values()))
    caches = {}
    for pid, path in file_map.items():
        cp = cache_dir / f'{pid}.npz'
        source_build_cache_for_participant(path, cp, mode=mode, common_c=common_c)
        caches[pid] = source_load_cache(cp)
    return caches

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def atomic_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{RUN_PART_NAME}.{os.getpid()}')
    compression = 'gzip' if str(path).endswith('.gz') else None
    df.to_csv(tmp, index=False, compression=compression)
    os.replace(tmp, path)

def atomic_json(obj: Mapping[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{RUN_PART_NAME}.{os.getpid()}')
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    os.replace(tmp, path)

def try_mount_drive() -> None:
    if Path('/content/gdrive/MyDrive').exists():
        return
    try:
        from google.colab import drive
        drive.mount('/content/gdrive', force_remount=False)
    except Exception:
        pass

def find_root(explicit: Optional[str]=None) -> Path:
    try_mount_drive()
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    candidates += [Path('/content/gdrive/MyDrive/HonorProject/HemoBoundary/SHIN'), Path('/content/gdrive/MyDrive/HonorProject/HemoBoundary/SHIN')]
    for p in candidates:
        if (p / '02_preprocessed').is_dir():
            return p
    raise FileNotFoundError('Could not find SHIN root containing 02_preprocessed. Use --shin-root.')

def add_pid_weight(cache):
    out = {}
    for pid, item in cache.items():
        out[pid] = {}
        for split, d0 in item.items():
            d = {k: np.asarray(v) for k, v in d0.items()}
            n = len(d['present'])
            d['pid'] = np.full(n, pid, dtype='U5')
            d['weight'] = np.ones(n, dtype=float)
            out[pid][split] = d
    return out

def training_rows(hb, caches, pids):
    return hb.cat([caches[p]['train'] for p in sorted(pids)])


In [ ]:
# Cell 3: Semi-synthetic bank construction

from __future__ import annotations

def source_stable_seed(*parts: Any) -> int:
    raw = '|'.join((str(x) for x in parts)).encode('utf-8')
    return int(hashlib.sha256(raw).hexdigest()[:8], 16) % (2 ** 31 - 1)

def source_ar_noise(scale: np.ndarray, phi: float, rng: np.random.Generator, n_time: int) -> np.ndarray:
    C = len(scale)
    x = np.zeros((C, n_time), float)
    common = rng.normal(size=n_time)
    for k in range(n_time):
        eps = 0.75 * rng.normal(size=C) + 0.25 * common[k]
        if k == 0:
            x[:, k] = eps
        else:
            x[:, k] = phi * x[:, k - 1] + math.sqrt(max(1.0 - phi ** 2, 1e-06)) * eps
    return x * scale[:, None]

def source_piecewise_shape(onset: float, rise: float, active: float, decay: float) -> Tuple[np.ndarray, float, float, np.ndarray]:
    peak = onset + rise
    active_end = peak + active
    offset = active_end + decay
    y = np.zeros(len(source_GRID), float)
    states = np.zeros(len(source_GRID), int)
    for i, tt in enumerate(source_GRID):
        if tt < onset:
            y[i] = 0.0
            states[i] = 0
        elif tt < peak:
            u = (tt - onset) / max(rise, 1e-08)
            y[i] = 0.5 - 0.5 * np.cos(np.pi * u)
            states[i] = 1
        elif tt < active_end:
            y[i] = 1.0
            states[i] = 2
        elif tt < offset:
            u = (tt - active_end) / max(decay, 1e-08)
            y[i] = 0.5 + 0.5 * np.cos(np.pi * u)
            states[i] = 3
        else:
            y[i] = 0.0
            states[i] = 4
    return (y, peak, offset, states)

def source_synthetic_trial(profile: NoiseProfile, rng: np.random.Generator, positive: bool) -> Dict[str, Any]:
    H = source_ar_noise(profile.hbo_scale, profile.phi_hbo, rng, len(source_FULL_T))
    R = source_ar_noise(profile.hbr_scale, profile.phi_hbr, rng, len(source_FULL_T))
    systemic = rng.uniform(0.0, 0.8) * np.nanmedian(profile.hbo_scale) * np.sin(2 * np.pi * rng.uniform(0.06, 0.12) * source_FULL_T + rng.uniform(0, 2 * np.pi))
    H += systemic[None, :]
    R += rng.uniform(0.5, 1.0) * systemic[None, :]
    if not positive:
        return dict(hbo=H, hbr=R, present=0, onset=np.nan, peak=np.nan, offset=np.nan, censored=0, clean_shape=np.zeros(len(source_GRID), float), states=np.zeros(len(source_GRID), int))
    onset = float(rng.uniform(0.5, 4.5))
    rise = float(rng.uniform(1.0, 3.5))
    active = float(rng.uniform(0.5, 3.0))
    if rng.random() < 0.18:
        decay = float(rng.uniform(7.0, 12.0))
    else:
        decay = float(rng.uniform(2.0, 8.0))
    shape, peak, offset, states = source_piecewise_shape(onset, rise, active, decay)
    post_shape = np.zeros(len(source_FULL_T), float)
    post_shape[source_POST_MASK] = shape
    support = rng.uniform(0.55, 1.0, size=profile.n_channels)
    amp = rng.uniform(1.5, 4.5) * profile.hbo_scale
    H += (amp * support)[:, None] * post_shape[None, :]
    R -= (amp * rng.uniform(0.35, 0.7) * rng.uniform(0.7, 1.1, size=profile.n_channels))[:, None] * post_shape[None, :]
    return dict(hbo=H, hbr=R, present=1, onset=onset, peak=peak, offset=offset, censored=int(offset > source_GRID_END - 0.25), clean_shape=shape, states=states)

def source_build_cache_for_participant(path: Path, cache_path: Path, mode: str, common_c: int=0) -> None:
    if cache_path.is_file():
        return
    profile = source_estimate_noise_profile(path)
    if common_c > 0 and profile.n_channels > common_c:
        profile = source_NoiseProfile(common_c, profile.hbo_scale[:common_c], profile.hbr_scale[:common_c], profile.phi_hbo, profile.phi_hbr)

    def make(split: str, npos: int, nneg: int) -> Dict[str, np.ndarray]:
        rng = np.random.default_rng(source_stable_seed('RAW_D_SCREEN', path.stem, split))
        trials = [source_synthetic_trial(profile, rng, True) for _ in range(npos)]
        trials += [source_synthetic_trial(profile, rng, False) for _ in range(nneg)]
        rng.shuffle(trials)
        out: Dict[str, List[Any]] = dict(present=[], onset=[], peak=[], offset=[], censored=[], clean=[], states=[], obs=[], state_feat=[], ch_anti=[])
        for tr in trials:
            ev = source_evidence(tr['hbo'], tr['hbr'])
            out['present'].append(tr['present'])
            out['onset'].append(tr['onset'])
            out['peak'].append(tr['peak'])
            out['offset'].append(tr['offset'])
            out['censored'].append(tr['censored'])
            out['clean'].append(tr['clean_shape'])
            out['states'].append(tr['states'])
            out['obs'].append(source_whole_vector(ev))
            out['state_feat'].append(source_state_features(ev))
            if mode == 'B':
                out['ch_anti'].append(source_channel_anti(ev, common_c))
        result = {'present': np.asarray(out['present'], np.int8), 'onset': np.asarray(out['onset'], float), 'peak': np.asarray(out['peak'], float), 'offset': np.asarray(out['offset'], float), 'censored': np.asarray(out['censored'], np.int8), 'clean': np.asarray(out['clean'], np.float32), 'states': np.asarray(out['states'], np.int8), 'obs': np.asarray(out['obs'], np.float32), 'state_feat': np.asarray(out['state_feat'], np.float32)}
        if mode == 'B':
            result['ch_anti'] = np.asarray(out['ch_anti'], np.float32)
        return result
    train = make('train', source_N_TRAIN_POS, source_N_TRAIN_NEG)
    test = make('test', source_N_TEST_POS, source_N_TEST_NEG)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {}
    for prefix, d in [('tr', train), ('te', test)]:
        for k, v in d.items():
            arrays[f'{prefix}_{k}'] = v
    np.savez_compressed(cache_path, **arrays)


In [ ]:
# Cell 4: Feature extraction

from __future__ import annotations

def source_baseline_z(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, float)
    bm = source_FULL_T < 0
    med = np.nanmedian(arr[:, bm], axis=1, keepdims=True)
    mad = 1.4826 * np.nanmedian(np.abs(arr[:, bm] - med), axis=1, keepdims=True)
    sd = np.nanstd(arr[:, bm], axis=1, keepdims=True)
    scale = np.where(np.isfinite(mad) & (mad > 1e-08), mad, np.where(sd > 1e-08, sd, 1.0))
    return (arr - med) / scale

def source_evidence(hbo: np.ndarray, hbr: np.ndarray) -> Dict[str, np.ndarray]:
    H = source_baseline_z(hbo)[:, source_POST_MASK]
    R = source_baseline_z(hbr)[:, source_POST_MASK]
    ghbo = np.nanmedian(H, axis=0)
    ghbr = np.nanmedian(R, axis=0)
    anti = 0.5 * (ghbo - ghbr)
    dhbo = np.gradient(ghbo, source_GRID_DT)
    dhbr = np.gradient(ghbr, source_GRID_DT)
    danti = np.gradient(anti, source_GRID_DT)
    dH = np.gradient(H, source_GRID_DT, axis=1)
    dR = np.gradient(R, source_GRID_DT, axis=1)
    cons_hbo = np.mean(np.sign(dH) == np.sign(dhbo)[None, :], axis=0)
    cons_hbr = np.mean(np.sign(dR) == np.sign(dhbr)[None, :], axis=0)
    iqr_hbo = np.nanquantile(H, 0.75, axis=0) - np.nanquantile(H, 0.25, axis=0)
    iqr_hbr = np.nanquantile(R, 0.75, axis=0) - np.nanquantile(R, 0.25, axis=0)
    return dict(H=H, R=R, ghbo=ghbo, ghbr=ghbr, anti=anti, dhbo=dhbo, dhbr=dhbr, danti=danti, cons_hbo=cons_hbo, cons_hbr=cons_hbr, iqr_hbo=iqr_hbo, iqr_hbr=iqr_hbr)

def source_whole_vector(ev: Mapping[str, np.ndarray]) -> np.ndarray:
    return np.concatenate([ev['ghbo'], ev['ghbr'], ev['anti'], ev['danti']]).astype(np.float32)

def source_state_features(ev: Mapping[str, np.ndarray]) -> np.ndarray:
    return np.column_stack([ev['ghbo'], ev['ghbr'], ev['anti'], ev['dhbo'], ev['dhbr'], ev['danti'], ev['cons_hbo'], ev['cons_hbr'], 0.5 * (ev['iqr_hbo'] + ev['iqr_hbr'])]).astype(np.float32)

def source_channel_anti(ev: Mapping[str, np.ndarray], common_c: int) -> np.ndarray:
    H = ev['H'][:common_c]
    R = ev['R'][:common_c]
    return (0.5 * (H - R)).astype(np.float32)

def source_finite_model_array(x: np.ndarray) -> np.ndarray:
    a = np.asarray(x)
    if not np.issubdtype(a.dtype, np.number):
        return a
    if np.all(np.isfinite(a)):
        return a
    return np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0, copy=True)

def source_assert_finite_model_inputs(data: Mapping[str, np.ndarray], context: str) -> None:
    bad = {}
    for key in source_FEATURE_ARRAY_KEYS:
        if key not in data:
            continue
        a = np.asarray(data[key])
        if np.issubdtype(a.dtype, np.number):
            nbad = int(np.size(a) - np.sum(np.isfinite(a)))
            if nbad:
                bad[key] = nbad
    if bad:
        raise RuntimeError(f'{context}: non-finite model inputs remained after sanitization: {bad}')

def hb_stable_seed(*parts: Any) -> int:
    return int(hashlib.sha256('|'.join(map(str, parts)).encode()).hexdigest()[:8], 16) % (2 ** 31 - 1)

def hb_finite_model_array(x: np.ndarray) -> np.ndarray:
    return np.nan_to_num(np.asarray(x, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

def hb_predictor_view(data: Mapping[str, np.ndarray]) -> Dict[str, np.ndarray]:
    return {k: data[k] for k in hb_PREDICTOR_KEYS}

def hb_require_predictors_only(x: Mapping[str, np.ndarray]) -> None:
    if set(x) != set(hb_PREDICTOR_KEYS):
        raise RuntimeError(f'Prediction payload must contain ONLY {hb_PREDICTOR_KEYS}; got {sorted(x)}')

def hb_grouped_folds(pids: Sequence[str], seed: int, n: int=3) -> List[List[str]]:
    p = np.array(sorted(set(pids)), dtype=object)
    if len(p) < 3:
        raise RuntimeError('At least three source participants are required.')
    np.random.default_rng(seed).shuffle(p)
    return [q.tolist() for q in np.array_split(p, min(n, len(p))) if len(q)]

def hb_subset(data: Mapping[str, np.ndarray], mask: np.ndarray) -> Dict[str, np.ndarray]:
    return {k: np.asarray(v)[mask] for k, v in data.items()}

def hb_cat(parts: Sequence[Mapping[str, np.ndarray]]) -> Dict[str, np.ndarray]:
    return {k: np.concatenate([p[k] for p in parts], axis=0) for k in parts[0]}

class hb_FitAudit:

    def __init__(self, allowed: Sequence[str], forbidden: Sequence[str], context: str):
        self.allowed = set(allowed)
        self.forbidden = set(forbidden)
        self.context = context
        self.rows = []

    def record(self, data: Mapping[str, np.ndarray], stage: str, held: Sequence[str]=()):
        actual = set(map(str, data['pid']))
        blocked = self.forbidden | set(held)
        if not actual or not actual.issubset(self.allowed) or actual & blocked:
            raise RuntimeError(f'{self.context}/{stage}: invalid fit membership {actual & blocked}')
        self.rows.append(dict(context=self.context, stage=stage, fit_participants=sorted(actual), forbidden=sorted(blocked), n_rows=int(len(data['pid']))))

def hb_smooth_series(x: np.ndarray, window: int) -> np.ndarray:
    x = np.asarray(x, float)
    if int(window) <= 1:
        return x
    w = min(int(window), len(x) if len(x) % 2 == 1 else len(x) - 1)
    if w < 3:
        return x
    if w % 2 == 0:
        w -= 1
    return signal.savgol_filter(x, w, 2 if w >= 5 else 1, mode='interp')

def hb_change_branch_features(data: Mapping[str, np.ndarray], frac: float) -> np.ndarray:
    F = hb_finite_model_array(data['state_feat']).astype(float, copy=False)
    anti = F[:, :, 2]
    rows: List[List[float]] = []
    for a in anti:
        scale_rows = []
        for w in hb_CHANGE_WINDOWS:
            z = hb_smooth_series(a, int(w))
            dz = np.gradient(z, hb_GRID_DT)
            ddz = np.gradient(dz, hb_GRID_DT)
            early = np.flatnonzero(hb_GRID <= 8.0)
            oi = int(early[np.argmax(dz[early])]) if len(early) else int(np.argmax(dz))
            pi = int(oi + np.argmax(z[oi:]))
            mx = float(z[pi])
            aft = np.arange(pi + 1, len(z))
            offi = len(z) - 1
            if len(aft) and mx > 0:
                cand = aft[z[aft] <= float(frac) * mx]
                if len(cand):
                    offi = int(cand[0])
            feat = [float(hb_GRID[oi]), float(hb_GRID[pi]), float(hb_GRID[offi]), float(np.max(dz)), float(np.min(dz)), mx, float(np.max(ddz)), float(np.min(ddz)), float(np.mean(np.abs(dz)))]
            scale_rows.append(feat)
        A = np.asarray(scale_rows, float)
        flat = A.reshape(-1).tolist()
        for col in (0, 1, 2):
            x = A[:, col]
            flat.extend([float(np.median(x)), float(np.quantile(x, 0.75) - np.quantile(x, 0.25)), float(np.max(x) - np.min(x))])
        flat.extend([float(np.mean(A[:, 3])), float(np.mean(A[:, 4])), float(np.mean(A[:, 5])), float(np.std(A[:, 3])), float(np.std(A[:, 4]))])
        rows.append(flat)
    return hb_finite_model_array(np.asarray(rows, float))

def hb_enforce_geometry(on: np.ndarray, pk: np.ndarray, off: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    on = np.asarray(on, float).copy()
    pk = np.asarray(pk, float).copy()
    off = np.asarray(off, float).copy()
    on = np.clip(on, 0.0, hb_GRID_END - 2 * hb_MIN_GAP)
    pk = np.maximum(pk, on + hb_MIN_GAP)
    pk = np.clip(pk, hb_MIN_GAP, hb_GRID_END - hb_MIN_GAP)
    off = np.maximum(off, pk + hb_MIN_GAP)
    off = np.clip(off, 2 * hb_MIN_GAP, hb_GRID_END)
    return (on, pk, off)

def hb_balanced_accuracy(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y, int)
    p = np.asarray(p, int)
    pos, neg = (y == 1, y == 0)
    sens = np.mean(p[pos] == 1) if np.any(pos) else np.nan
    spec = np.mean(p[neg] == 0) if np.any(neg) else np.nan
    return float(np.nanmean([sens, spec]))

def hb_trial_metric_summary(truth: Mapping[str, np.ndarray], pred: pd.DataFrame) -> Dict[str, float]:
    present = np.asarray(truth['present'], int) == 1
    cens = np.asarray(truth['censored'], int) == 1
    decision = pred['pred_response'].to_numpy(int)
    out: Dict[str, float] = {'response_balanced_accuracy': hb_balanced_accuracy(np.asarray(truth['present'], int), decision)}
    metrics = {}
    for b, tcol, pcol in [('onset', 'onset', 'pred_onset'), ('peak', 'peak', 'pred_peak'), ('offset', 'offset', 'pred_offset')]:
        t = np.asarray(truth[tcol], float)
        p = pred[pcol].to_numpy(float)
        valid_truth = present & np.isfinite(t)
        if b == 'offset':
            valid_truth &= ~cens
        err = np.full(np.sum(valid_truth), hb_MISS_PENALTY_SEC, float)
        idx = np.flatnonzero(valid_truth)
        finite = np.isfinite(p[idx]) & (decision[idx] == 1)
        err[finite] = np.abs(p[idx][finite] - t[idx][finite])
        metrics[f'{b}_mae_sec'] = float(np.mean(err)) if len(err) else np.nan
        metrics[f'{b}_localized_rate'] = float(np.mean(finite)) if len(err) else np.nan
    ton = np.asarray(truth['onset'], float)
    toff = np.asarray(truth['offset'], float)
    pon = pred['pred_onset'].to_numpy(float)
    poff = pred['pred_offset'].to_numpy(float)
    valid = present & ~cens & np.isfinite(ton) & np.isfinite(toff)
    idx = np.flatnonzero(valid)
    err = np.full(len(idx), hb_MISS_PENALTY_SEC, float)
    finite = np.isfinite(pon[idx]) & np.isfinite(poff[idx]) & (decision[idx] == 1)
    err[finite] = np.abs(poff[idx][finite] - pon[idx][finite] - (toff[idx][finite] - ton[idx][finite]))
    metrics['duration_mae_sec'] = float(np.mean(err)) if len(err) else np.nan
    out.update(metrics)
    scales = np.array([1.0, 1.0, 1.5, 1.5])
    vals = np.array([out['onset_mae_sec'], out['peak_mae_sec'], out['offset_mae_sec'], out['duration_mae_sec']])
    out['selection_loss'] = float(np.nanmean(vals / scales))
    return out


In [ ]:
# Cell 5: Temporal Boundary Estimation

from __future__ import annotations

class hb_StableScaler:

    def fit(self, X, sample_weight=None):
        X = np.asarray(X, float)
        if X.ndim != 2 or not np.isfinite(X).all():
            raise RuntimeError('Scaler needs finite 2D training features.')
        w = np.ones(len(X)) if sample_weight is None else np.asarray(sample_weight, float)
        if len(w) != len(X) or not np.isfinite(w).all() or (w < 0).any() or (w.sum() <= 0):
            raise RuntimeError('Invalid source sample weights.')
        self.mean_ = np.average(X, axis=0, weights=w)
        self.var_ = np.average((X - self.mean_) ** 2, axis=0, weights=w)
        self.var_ = np.maximum(self.var_, 0.0)
        eps = np.finfo(float).eps
        n = float(w.sum())
        constant = self.var_ <= n * eps * self.var_ + (n * self.mean_ * eps) ** 2
        self.scale_ = np.sqrt(self.var_)
        self.scale_[constant | (self.scale_ == 0)] = 1.0
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = np.asarray(X, float)
        if X.ndim != 2 or X.shape[1] != self.n_features_in_ or (not np.isfinite(X).all()):
            raise RuntimeError('Scaler feature schema/finite check failed.')
        return (X - self.mean_) / self.scale_

@dataclass
class hb_StateModel:
    scaler: Any
    clf: Any
    response_scaler: Any
    response_clf: Any
    fit_pids: Tuple[str, ...]

def hb_fixed_order_path(p, stay_bias=0.0):
    logp = np.log(np.clip(p, 1e-09, 1.0))
    n, t, s = logp.shape
    dp = np.full((n, s), -np.inf)
    dp[:, 0] = logp[:, 0, 0]
    back = np.zeros((n, t, s), np.int8)
    for k in range(1, t):
        nxt = np.full_like(dp, -np.inf)
        for state in range(s):
            vs = dp[:, state] + stay_bias
            va = dp[:, state - 1] if state > 0 else np.full(n, -np.inf)
            keep = vs >= va
            nxt[:, state] = np.where(keep, vs, va) + logp[:, k, state]
            back[:, k, state] = np.where(keep, state, state - 1)
        dp = nxt
    states = np.full(n, s - 1, dtype=int)
    path = np.zeros((n, t), np.int8)
    ii = np.arange(n)
    for k in range(t - 1, -1, -1):
        path[:, k] = states
        if k > 0:
            states = back[ii, k, states]
    return path

def hb_hard_state_features(p, r):
    n, t, s = p.shape
    path = hb_fixed_order_path(p)
    occ = p.mean(axis=1)
    mx = p.max(axis=1)
    centres = (p * hb_GRID[None, :, None]).sum(axis=1) / np.maximum(p.sum(axis=1), 1e-12)
    summaries = np.stack((occ, mx, centres), axis=2).reshape(n, 15)
    dwell = np.column_stack([(path == state).mean(axis=1) for state in range(5)])
    oi = np.argmax(path == 1, axis=1)
    pi = np.argmax(np.where(path == 2, p[:, :, 2], -np.inf), axis=1)
    fi = np.argmax(path == 4, axis=1)
    ii = np.arange(n)
    times = np.column_stack((hb_GRID[oi], hb_GRID[pi], hb_GRID[fi]))
    conf = np.column_stack((p[ii, oi, 1], p[ii, pi, 2], p[ii, fi, 4]))
    ent = -(p * np.log(np.clip(p, 1e-09, 1))).sum(axis=2)
    return np.column_stack((summaries, dwell, times, conf, ent.mean(axis=1), ent.max(axis=1), r, (p[:, :, 1] + p[:, :, 2]).mean(axis=1), (p[:, :, 3] + p[:, :, 4]).mean(axis=1)))

@dataclass
class hb_Heads:
    scaler: Any
    response: Any
    regressors: Any
    coefficients: Any = None

    def raw(self, X):
        Z = self.scaler.transform(X)
        prob = self.response.predict_proba(Z)[:, 1]
        if self.coefficients is None:
            t = np.column_stack([r.predict(Z) for r in self.regressors])
        else:
            t = np.column_stack((Z, np.ones(len(Z)))) @ self.coefficients
        if not np.isfinite(t).all():
            raise RuntimeError('Non-finite timing-head prediction.')
        return (t, prob)

def hb_direct_heads(X, data, alpha, audit):
    audit.record(data, 'boundary_and_response_heads')
    w = np.asarray(data['weight'], float)
    w = w / w.mean()
    sc = hb_StableScaler().fit(X, sample_weight=w)
    Z = sc.transform(X)
    resp = LogisticRegression(C=0.5, class_weight='balanced', max_iter=1000, random_state=hb_RANDOM_SEED)
    resp.fit(Z, np.asarray(data['present'], int), sample_weight=w)
    regs = []
    for b in ('onset', 'peak', 'offset'):
        truth = np.asarray(data[b], float)
        mask = (np.asarray(data['present']) == 1) & np.isfinite(truth)
        if b == 'offset':
            mask &= np.asarray(data['censored']) == 0
        if mask.sum() < 15:
            raise RuntimeError(f'Too few observed positive labels for {b}.')
        rw = w[mask] / w[mask].mean()
        regs.append(Ridge(alpha=float(alpha)).fit(Z[mask], truth[mask], sample_weight=rw))
    return hb_Heads(sc, resp, regs)

def base_stable_seed(*parts: Any) -> int:
    raw = '|'.join((str(x) for x in parts)).encode('utf-8')
    return int(hashlib.sha256(raw).hexdigest()[:8], 16) % (2 ** 31 - 1)

def stable_seed(*parts: Any) -> int:
    if int(ACTIVE_RUN_SEED) == 1:
        return base_stable_seed(*parts)
    return base_stable_seed('MULTISEED_REPEAT', int(ACTIVE_RUN_SEED), *parts)

def canonical_cfg(cfg: Mapping[str, Any]) -> str:
    return json.dumps({k: cfg[k] for k in sorted(cfg)}, sort_keys=True, separators=(',', ':'))

def make_prediction_threshold(hb, raw, prob, threshold):
    raw = np.asarray(raw, float)
    prob = np.asarray(prob, float)
    on, pk, off = hb.enforce_geometry(raw[:, 0], raw[:, 1], raw[:, 2])
    miss = prob < float(threshold)
    on = on.astype(float)
    pk = pk.astype(float)
    off = off.astype(float)
    on[miss] = np.nan
    pk[miss] = np.nan
    off[miss] = np.nan
    return pd.DataFrame({'pred_response': (~miss).astype(int), 'pred_onset': on, 'pred_peak': pk, 'pred_offset': off})

def coherent_canonical_hrf(t: np.ndarray, scale: float=1.0) -> np.ndarray:
    t = np.asarray(t, float)
    x = np.clip(t / float(scale), 0.0, None)
    h = scipy_gamma.pdf(x, 6.0) - scipy_gamma.pdf(x, 16.0) / 6.0
    h[t < 0] = 0.0
    mx = float(np.max(np.abs(h)))
    if mx > 0:
        h = h / mx
    return h

def coherent_glm_basis(grid: np.ndarray) -> np.ndarray:
    grid = np.asarray(grid, float)
    h = coherent_canonical_hrf(grid, 1.0)
    dt = float(np.median(np.diff(grid)))
    temporal = np.gradient(h, dt)
    eps = 0.08
    wide = coherent_canonical_hrf(grid, 1.0 + eps)
    narrow = coherent_canonical_hrf(grid, 1.0 - eps)
    dispersion = (wide - narrow) / (2.0 * eps)
    B = np.column_stack([h, temporal, dispersion]).astype(float)
    for j in range(B.shape[1]):
        nrm = float(np.sqrt(np.sum(B[:, j] ** 2)))
        if nrm > 1e-12:
            B[:, j] /= nrm
    return B

def coherent_anti(hb, x: Mapping[str, np.ndarray]) -> np.ndarray:
    hb.require_predictors_only(x)
    F = hb.finite_model_array(x['state_feat']).astype(float, copy=False)
    if F.ndim != 3 or F.shape[1] != len(hb.GRID) or F.shape[2] < 3:
        raise RuntimeError(f'Unexpected state_feat shape {F.shape}.')
    return np.asarray(F[:, :, 2], float)

def coherent_glm_features_times(hb, x: Mapping[str, np.ndarray]):
    Y = coherent_anti(hb, x)
    grid = np.asarray(hb.GRID, float)
    B = coherent_glm_basis(grid)
    trend = (grid - np.mean(grid)) / max(float(np.std(grid)), 1e-12)
    X = np.column_stack([B, np.ones(len(grid)), trend])
    penalty = np.diag([BASE_GLM_RIDGE, BASE_GLM_RIDGE, BASE_GLM_RIDGE, 0.0, 0.0])
    projection = np.linalg.pinv(X.T @ X + penalty) @ X.T
    feats, times = ([], [])
    for y in Y:
        beta = projection @ np.asarray(y, float)
        task = B @ beta[:3]
        fitted = X @ beta
        resid = np.asarray(y, float) - fitted
        sse = float(np.sum(resid ** 2))
        centred = np.asarray(y, float) - float(np.mean(y))
        sst = float(np.sum(centred ** 2))
        r2 = 1.0 - sse / max(sst, 1e-12)
        peak_idx = int(np.argmax(task))
        peak_val = float(task[peak_idx])
        floor = float(np.min(task[:peak_idx + 1]))
        amp = max(peak_val - floor, 1e-08)
        level = floor + BASE_BOUNDARY_FRACTION * amp
        pre = np.flatnonzero(task[:peak_idx + 1] >= level)
        on_idx = int(pre[0]) if len(pre) else 0
        post = np.flatnonzero(task[peak_idx + 1:] <= level)
        off_idx = int(peak_idx + 1 + post[0]) if len(post) else len(grid) - 1
        on, pk, off = hb.enforce_geometry(np.asarray([grid[on_idx]], float), np.asarray([grid[peak_idx]], float), np.asarray([grid[off_idx]], float))
        positive = np.clip(task, 0.0, None)
        feats.append([float(beta[0]), float(beta[1]), float(beta[2]), float(r2), float(peak_val), float(np.sum(positive)), float(np.mean(np.abs(resid))), float(np.std(resid)), float(np.max(y)), float(np.min(y))])
        times.append([float(on[0]), float(pk[0]), float(off[0])])
    return (hb.finite_model_array(np.asarray(feats, float)), np.asarray(times, float))

@dataclass
class CoherentBaseModel:
    hb: Any
    scaler: Any
    response_clf: Any

    def raw(self, x):
        feat, raw = coherent_glm_features_times(self.hb, x)
        prob = self.response_clf.predict_proba(self.scaler.transform(feat))[:, 1]
        return (raw, np.asarray(prob, float))

    def predict(self, x):
        raw, prob = self.raw(x)
        return make_prediction_threshold(self.hb, raw, prob, BASE_RESPONSE_THRESHOLD)

def fit_coherent_base(hb, data, audit, stage_prefix='BASE'):
    audit.record(data, stage_prefix + '_response_classifier')
    feat, _ = coherent_glm_features_times(hb, hb.predictor_view(data))
    y = np.asarray(data['present'], int)
    w = np.asarray(data['weight'], float)
    scaler = hb.StableScaler().fit(feat, sample_weight=w)
    clf = LogisticRegression(C=BASE_RESPONSE_C, class_weight='balanced', max_iter=1000, random_state=hb.RANDOM_SEED)
    clf.fit(scaler.transform(feat), y, sample_weight=w / np.mean(w))
    return CoherentBaseModel(hb=hb, scaler=scaler, response_clf=clf)

@dataclass
class CoherentStateModel:
    scaler: Any
    clf: Any

def fit_coherent_state(hb, data, state_C, audit, stage_prefix):
    audit.record(data, stage_prefix + '_state_encoder')
    F = hb.finite_model_array(data['state_feat']).astype(float, copy=False)
    X = F.reshape(-1, F.shape[-1])
    y = np.asarray(data['states'], int).reshape(-1)
    w = np.asarray(data['weight'], float)
    fw = np.repeat(w, F.shape[1])
    fw = fw / np.mean(fw)
    if len(X) > 60000:
        idx = np.random.default_rng(stable_seed('COHERENT_STATE_SUBSAMPLE', stage_prefix, *sorted(set(map(str, data['pid']))), float(state_C))).choice(len(X), 60000, replace=False)
        X, y, fw = (X[idx], y[idx], fw[idx])
    scaler = hb.StableScaler().fit(X, sample_weight=fw)
    clf = LogisticRegression(C=float(state_C), class_weight='balanced', max_iter=800, random_state=hb.RANDOM_SEED)
    clf.fit(scaler.transform(X), y, sample_weight=fw)
    return CoherentStateModel(scaler=scaler, clf=clf)

def coherent_state_probabilities(hb, model, x):
    hb.require_predictors_only(x)
    F = hb.finite_model_array(x['state_feat']).astype(float, copy=False)
    n, t, d = F.shape
    p0 = model.clf.predict_proba(model.scaler.transform(F.reshape(-1, d)))
    P = np.full((n * t, 5), 1e-08, float)
    for j, cls in enumerate(model.clf.classes_):
        P[:, int(cls)] = p0[:, j]
    P /= np.maximum(P.sum(axis=1, keepdims=True), 1e-12)
    return P.reshape(n, t, 5)

def coherent_state_features(hb, P):
    zero_response = np.zeros(len(P), float)
    full = hb.hard_state_features(P, zero_response)
    if full.shape[1] != 31:
        raise RuntimeError(f'Unexpected hard_state_features width {full.shape[1]} (expected 31).')
    return np.column_stack([full[:, :28], full[:, 29:]])

@dataclass
class CoherentTimingHeads:
    scaler: Any
    regs: Dict[str, Any]
    names: Tuple[str, ...]

    def raw(self, X):
        Z = self.scaler.transform(X)
        return {name: self.regs[name].predict(Z) for name in self.names}

def fit_coherent_timing_heads(hb, X, data, alpha, names, audit, stage_prefix):
    audit.record(data, stage_prefix + '_timing_heads')
    names = tuple(names)
    w = np.asarray(data['weight'], float)
    scaler = hb.StableScaler().fit(X, sample_weight=w)
    Z = scaler.transform(X)
    present = np.asarray(data['present'], int) == 1
    censored = np.asarray(data['censored'], int) == 1
    regs = {}
    for name in names:
        y = np.asarray(data[name], float)
        mask = present & np.isfinite(y)
        if name == 'offset':
            mask &= ~censored
        if int(np.sum(mask)) < 15:
            raise RuntimeError(f'{stage_prefix}: too few rows for {name}.')
        rw = w[mask] / np.mean(w[mask])
        regs[name] = Ridge(alpha=float(alpha)).fit(Z[mask], y[mask], sample_weight=rw)
    return CoherentTimingHeads(scaler=scaler, regs=regs, names=names)

@dataclass
class CoherentAModel:
    hb: Any
    state: CoherentStateModel
    heads: CoherentTimingHeads

    def raw(self, x):
        P = coherent_state_probabilities(self.hb, self.state, x)
        X = coherent_state_features(self.hb, P)
        pred = self.heads.raw(X)
        return (np.asarray(pred['onset'], float), np.asarray(pred['peak'], float), P)

def fit_coherent_A(hb, data, cfg, audit, stage_prefix='A'):
    state = fit_coherent_state(hb, data, float(cfg['state_C']), audit, stage_prefix)
    P = coherent_state_probabilities(hb, state, hb.predictor_view(data))
    X = coherent_state_features(hb, P)
    heads = fit_coherent_timing_heads(hb, X, data, cfg['timing_alpha'], ('onset', 'peak'), audit, stage_prefix)
    return CoherentAModel(hb=hb, state=state, heads=heads)

def predict_row_A(hb, base, a_model, x):
    base_raw, base_prob = base.raw(x)
    on, pk, _P = a_model.raw(x)
    raw = np.column_stack([on, pk, base_raw[:, 2]])
    on2, pk2, off2 = hb.enforce_geometry(raw[:, 0], raw[:, 1], raw[:, 2])
    raw = np.column_stack([on2, pk2, off2])
    return make_prediction_threshold(hb, raw, base_prob, BASE_RESPONSE_THRESHOLD)


In [ ]:
# Cell 6: Recovery Estimation

from __future__ import annotations

def coherent_ab_features(hb, state_model, x, change_frac):
    P = coherent_state_probabilities(hb, state_model, x)
    SX = coherent_state_features(hb, P)
    CX = hb.change_branch_features(x, float(change_frac))
    return (hb.finite_model_array(np.column_stack([SX, CX])), P)

@dataclass
class CoherentABModel:
    hb: Any
    state: CoherentStateModel
    heads: CoherentTimingHeads
    change_frac: float

    def raw(self, x):
        X, P = coherent_ab_features(self.hb, self.state, x, self.change_frac)
        pred = self.heads.raw(X)
        on, pk, off = self.hb.enforce_geometry(np.asarray(pred['onset'], float), np.asarray(pred['peak'], float), np.asarray(pred['offset'], float))
        return (np.column_stack([on, pk, off]), P, X)

def fit_coherent_AB(hb, data, cfg, audit, stage_prefix='AB'):
    state = fit_coherent_state(hb, data, float(cfg['state_C']), audit, stage_prefix)
    X, _ = coherent_ab_features(hb, state, hb.predictor_view(data), float(cfg['change_frac']))
    heads = fit_coherent_timing_heads(hb, X, data, cfg['timing_alpha'], ('onset', 'peak', 'offset'), audit, stage_prefix)
    return CoherentABModel(hb=hb, state=state, heads=heads, change_frac=float(cfg['change_frac']))

def predict_row_AB(hb, base, ab_model, x):
    raw, _P, _X = ab_model.raw(x)
    _base_raw, base_prob = base.raw(x)
    return make_prediction_threshold(hb, raw, base_prob, BASE_RESPONSE_THRESHOLD)


In [ ]:
# Cell 7: Response Detection

from __future__ import annotations

@dataclass
class CoherentABCModel:
    hb: Any
    ab: CoherentABModel
    response_scaler: Any
    response_clf: Any
    response_threshold: float

    def raw(self, x):
        raw, P, X = self.ab.raw(x)
        prob = self.response_clf.predict_proba(self.response_scaler.transform(X))[:, 1]
        return (raw, np.asarray(prob, float), P)

    def predict(self, x):
        raw, prob, _ = self.raw(x)
        return make_prediction_threshold(self.hb, raw, prob, self.response_threshold)

def fit_coherent_ABC(hb, data, cfg, audit, stage_prefix='ABC'):
    ab = fit_coherent_AB(hb, data, cfg, audit, stage_prefix=stage_prefix + '_AB')
    X, _ = coherent_ab_features(hb, ab.state, hb.predictor_view(data), ab.change_frac)
    audit.record(data, stage_prefix + '_response_detector')
    w = np.asarray(data['weight'], float)
    y = np.asarray(data['present'], int)
    response_scaler = hb.StableScaler().fit(X, sample_weight=w)
    response_clf = LogisticRegression(C=0.5, class_weight='balanced', max_iter=1000, random_state=hb.RANDOM_SEED)
    response_clf.fit(response_scaler.transform(X), y, sample_weight=w / np.mean(w))
    return CoherentABCModel(hb=hb, ab=ab, response_scaler=response_scaler, response_clf=response_clf, response_threshold=float(cfg['response_threshold']))

def coherent_frame(target, row, truth, pred, selected_global='', selected_local='', inner_loss=np.nan):
    return pd.DataFrame({'participant': target, 'row': row, 'configuration': ROW_LABELS[row], 'meaning': ROW_MEANING[row], 'synthetic_index': np.arange(len(pred), dtype=int), 'truth_present': np.asarray(truth['present'], int), 'truth_onset': np.asarray(truth['onset'], float), 'truth_peak': np.asarray(truth['peak'], float), 'truth_offset': np.asarray(truth['offset'], float), 'truth_censored': np.asarray(truth['censored'], int), 'pred_response': pred['pred_response'].to_numpy(int), 'pred_onset': pred['pred_onset'].to_numpy(float), 'pred_peak': pred['pred_peak'].to_numpy(float), 'pred_offset': pred['pred_offset'].to_numpy(float), 'selected_global_cfg': selected_global, 'selected_local_cfg': selected_local, 'inner_selected_loss': float(inner_loss) if np.isfinite(inner_loss) else np.nan})


In [ ]:
# Cell 8: Local Boundary Refinement

from __future__ import annotations

def hb_local_profiles(x, probabilities, anchors, response_prob, radius):
    hb_require_predictors_only(x)
    F = hb_finite_model_array(x['state_feat'])
    P = np.asarray(probabilities, float)
    n = len(F)
    rel = np.linspace(-float(radius), float(radius), 9)
    out = []
    for b in range(3):
        rows = []
        for i in range(n):
            centre = float(anchors[i, b])
            t = centre + rel
            f = np.column_stack([np.interp(t, hb_GRID, F[i, :, c]) for c in range(F.shape[2])])
            p = np.column_stack([np.interp(t, hb_GRID, P[i, :, c]) for c in range(P.shape[2])])
            seen = ((t >= 0) & (t <= hb_GRID_END)).astype(float)
            context = np.r_[anchors[i], response_prob[i], anchors[i, 1] - anchors[i, 0], anchors[i, 2] - anchors[i, 1]]
            rows.append(np.r_[f.ravel(), p.ravel(), seen, context])
        out.append(np.asarray(rows, float))
    return out

def coherent_oof_anchors(hb, data, abc_cfg, outer_forbidden, audit_parent, context):
    pids = sorted(set(map(str, data['pid'])))
    n = len(data['pid'])
    raw = np.zeros((n, 3), float)
    rp = np.zeros(n, float)
    P = np.zeros((n, len(hb.GRID), 5), float)
    covered = np.zeros(n, int)
    folds = hb.grouped_folds(pids, stable_seed('COHERENT_LBR_OOF', context, canonical_cfg(abc_cfg), *pids), LOCAL_OOF_FOLDS_COHERENT)
    for held in folds:
        mask = np.isin(data['pid'], held)
        tr = hb.subset(data, ~mask)
        va = hb.subset(data, mask)
        tr_pids = sorted(set(map(str, tr['pid'])))
        audit = hb.FitAudit(tr_pids, set(outer_forbidden) | set(held), context + '/LBR_OOF')
        model = fit_coherent_ABC(hb, tr, abc_cfg, audit, stage_prefix='LBR_OOF_ABC')
        r, p, pp = model.raw(hb.predictor_view(va))
        raw[mask] = r
        rp[mask] = p
        P[mask] = pp
        covered[mask] += 1
        audit_parent.rows.extend(audit.rows)
    if not np.all(covered == 1):
        raise RuntimeError('Coherent LBR OOF anchor coverage failed.')
    return (raw, rp, P)

@dataclass
class CoherentFullModel:
    hb: Any
    abc: CoherentABCModel
    local_cfg: Dict[str, float]
    scalers: List[Any]
    regs: List[Any]

    def predict(self, x):
        raw, rp, P = self.abc.raw(x)
        profiles = self.hb.local_profiles(x, P, raw, rp, float(self.local_cfg['radius']))
        refined = np.asarray(raw, float).copy()
        for b in range(3):
            delta = self.regs[b].predict(self.scalers[b].transform(profiles[b]))
            radius = float(self.local_cfg['radius'])
            refined[:, b] += float(self.local_cfg['strength']) * np.clip(delta, -radius, radius)
        return make_prediction_threshold(self.hb, refined, rp, self.abc.response_threshold)

def fit_coherent_full(hb, data, abc_cfg, local_cfg, audit, outer_forbidden, context):
    abc = fit_coherent_ABC(hb, data, abc_cfg, audit, stage_prefix='FULL_ABC')
    raw_oof, rp_oof, P_oof = coherent_oof_anchors(hb, data, abc_cfg, outer_forbidden, audit, context)
    profiles = hb.local_profiles(hb.predictor_view(data), P_oof, raw_oof, rp_oof, float(local_cfg['radius']))
    audit.record(data, 'FULL_LBR_residual_heads_on_source_OOF_anchors')
    present = np.asarray(data['present'], int) == 1
    censored = np.asarray(data['censored'], int) == 1
    scalers, regs = ([], [])
    for b, name in enumerate(('onset', 'peak', 'offset')):
        y = np.asarray(data[name], float)
        mask = present & np.isfinite(y)
        if name == 'offset':
            mask &= ~censored
        scaler = hb.StableScaler().fit(profiles[b][mask])
        residual = y[mask] - raw_oof[mask, b]
        reg = Ridge(alpha=float(local_cfg['local_alpha'])).fit(scaler.transform(profiles[b][mask]), residual)
        scalers.append(scaler)
        regs.append(reg)
    return CoherentFullModel(hb=hb, abc=abc, local_cfg=dict(local_cfg), scalers=scalers, regs=regs)

SOURCE_NS = _EngineProxy("source_")
HB_NS = _EngineProxy("hb_")


In [ ]:
# Cell 9: Nested LOSO selection

from __future__ import annotations

def bootstrap_ci(x, seed):
    a = np.asarray(x, float)
    a = a[np.isfinite(a)]
    if len(a) == 0:
        return (np.nan, np.nan, np.nan)
    mean = float(np.mean(a))
    if len(a) == 1:
        return (mean, mean, mean)
    rng = np.random.default_rng(seed)
    vals = np.empty(BOOT_REPS, float)
    done = 0
    while done < BOOT_REPS:
        n = min(2000, BOOT_REPS - done)
        ix = rng.integers(0, len(a), size=(n, len(a)))
        vals[done:done + n] = np.mean(a[ix], axis=1)
        done += n
    lo, hi = np.quantile(vals, [0.025, 0.975])
    return (mean, float(lo), float(hi))

def signflip_p(x, seed):
    a = np.asarray(x, float)
    a = a[np.isfinite(a)]
    if len(a) == 0:
        return np.nan
    obs = abs(float(np.mean(a)))
    if obs < 1e-15:
        return 1.0
    rng = np.random.default_rng(seed)
    hits = 0
    done = 0
    while done < SIGNFLIP_REPS:
        n = min(2000, SIGNFLIP_REPS - done)
        signs = rng.choice([-1.0, 1.0], size=(n, len(a)))
        stat = np.abs(np.mean(signs * a[None, :], axis=1))
        hits += int(np.sum(stat >= obs - 1e-15))
        done += n
    return float((hits + 1) / (SIGNFLIP_REPS + 1))

def holm_adjust(pvals):
    p = np.asarray(pvals, float)
    out = np.full(len(p), np.nan, float)
    valid = np.flatnonzero(np.isfinite(p))
    if len(valid) == 0:
        return out
    order = valid[np.argsort(p[valid])]
    running = 0.0
    m = len(order)
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        out[idx] = min(1.0, running)
    return out

def run_model_checks(source, hb):
    assert len(GLOBAL_TIMING_GRID) == 7
    assert len(GLOBAL_RESPONSE_GRID) == 9
    assert len(LBR_GRID) == 7
    assert len(GLOBAL_RESPONSE_GRID) * len(LBR_GRID) == 63
    assert BASE_GLOBAL == {'state_C': 0.01, 'change_frac': 0.3, 'global_alpha': 10.0, 'response_threshold': 0.5}
    assert BASE_LBR == {'radius': 1.0, 'local_alpha': 100.0, 'strength': 0.5}
    vals = [(1.0, i) for i in range(3)]
    vals.sort()
    assert vals[0][1] == 0
    print('HemoBoundary model checks: PASS')

def mean_e2e_loss(hb, validation_by_pid, predictor):
    vals = []
    for pid, truth in validation_by_pid.items():
        pred = predictor(hb.predictor_view(truth))
        vals.append(float(hb.trial_metric_summary(truth, pred)['selection_loss']))
    return float(np.mean(vals))

def coherent_participant_metrics(hb, per_trial):
    rows = []
    for (pid, row), q in per_trial.groupby(['participant', 'row'], sort=False):
        truth = {'present': q['truth_present'].to_numpy(int), 'onset': q['truth_onset'].to_numpy(float), 'peak': q['truth_peak'].to_numpy(float), 'offset': q['truth_offset'].to_numpy(float), 'censored': q['truth_censored'].to_numpy(int)}
        pred = pd.DataFrame({'pred_response': q['pred_response'].to_numpy(int), 'pred_onset': q['pred_onset'].to_numpy(float), 'pred_peak': q['pred_peak'].to_numpy(float), 'pred_offset': q['pred_offset'].to_numpy(float)})
        m = hb.trial_metric_summary(truth, pred)
        rows.append({'participant': str(pid), 'row': str(row), 'configuration': ROW_LABELS[str(row)], 'meaning': ROW_MEANING[str(row)], **m})
    out = pd.DataFrame(rows)
    out['row'] = pd.Categorical(out['row'], categories=list(ROW_ORDER), ordered=True)
    return out.sort_values(['row', 'participant']).reset_index(drop=True)

def coherent_summary(pm):
    rows = []
    for row in ROW_ORDER:
        q = pm[pm['row'].astype(str) == row]
        if not len(q):
            continue
        r = {'row': row, 'configuration': ROW_LABELS[row], 'meaning': ROW_MEANING[row], 'n_participants': int(q['participant'].nunique())}
        for metric in ('response_balanced_accuracy', 'onset_mae_sec', 'peak_mae_sec', 'offset_mae_sec', 'duration_mae_sec', 'selection_loss'):
            r[metric] = float(np.nanmean(pd.to_numeric(q[metric], errors='coerce')))
        rows.append(r)
    return pd.DataFrame(rows)

def coherent_incremental_stats(pm):
    transitions = [(ROW_BASE, ROW_A, '+A'), (ROW_A, ROW_AB, '+B'), (ROW_AB, ROW_ABC, '+C'), (ROW_ABC, ROW_FULL, '+LBR')]
    rows = []
    for reference, candidate, label in transitions:
        a = pm[pm['row'].astype(str) == reference].set_index('participant')
        b = pm[pm['row'].astype(str) == candidate].set_index('participant')
        common = sorted(set(a.index) & set(b.index))
        for metric in ('response_balanced_accuracy', 'onset_mae_sec', 'peak_mae_sec', 'offset_mae_sec', 'duration_mae_sec', 'selection_loss'):
            av = a.loc[common, metric].to_numpy(float)
            bv = b.loc[common, metric].to_numpy(float)
            if metric == 'response_balanced_accuracy':
                gain = bv - av
            else:
                gain = av - bv
            mean, lo, hi = bootstrap_ci(gain, stable_seed('COHERENT_INCREMENTAL_BOOT', reference, candidate, metric))
            p = signflip_p(gain, stable_seed('COHERENT_INCREMENTAL_SIGN', reference, candidate, metric))
            rows.append({'transition': label, 'reference': reference, 'candidate': candidate, 'metric': metric, 'n_participants': len(common), 'mean_gain_positive_added_component_better': mean, 'ci95_low': lo, 'ci95_high': hi, 'participants_better': int(np.sum(gain > 1e-12)), 'participants_equal': int(np.sum(np.isclose(gain, 0.0))), 'participants_worse': int(np.sum(gain < -1e-12)), 'signflip_p': p})
    out = pd.DataFrame(rows)
    out['holm_p_4_timing_within_transition'] = np.nan
    timing = {'onset_mae_sec', 'peak_mae_sec', 'offset_mae_sec', 'duration_mae_sec'}
    for transition in out['transition'].drop_duplicates():
        ix = out.index[(out['transition'] == transition) & out['metric'].isin(timing)].tolist()
        out.loc[ix, 'holm_p_4_timing_within_transition'] = holm_adjust(out.loc[ix, 'signflip_p'].to_numpy(float))
    return out

def build_inner_contexts(hb, caches, outer_target, outer_source_pids):
    folds = hb.grouped_folds(outer_source_pids, stable_seed('STRICT_HIERARCHICAL_INNER', COHERENT_SEED, outer_target, *outer_source_pids), INNER_FOLDS)
    flat = sorted((p for fold in folds for p in fold))
    if flat != sorted(outer_source_pids):
        raise RuntimeError(f'{outer_target}: inner participant coverage failed.')
    contexts = []
    for fi, held_pids in enumerate(folds, 1):
        tr_pids = [p for p in outer_source_pids if p not in held_pids]
        contexts.append({'fold': fi, 'held_pids': list(held_pids), 'train_pids': tr_pids, 'train': training_rows(hb, caches, tr_pids), 'validation_by_pid': {pid: caches[pid]['test'] for pid in held_pids}, 'forbidden': set(held_pids) | {outer_target}})
    return contexts

def choose_configuration(score_dict, ordered_candidates):
    ranked = []
    for idx, candidate in enumerate(ordered_candidates):
        key = canonical_cfg(candidate)
        vals = score_dict[key]
        if not len(vals):
            raise RuntimeError('Hierarchical candidate has no validation scores.')
        ranked.append((float(np.mean(vals)), idx, dict(candidate)))
    ranked.sort(key=lambda x: (x[0], x[1]))
    return (ranked[0][2], float(ranked[0][0]))

def select_coherent_configs(hb, caches, outer_target, outer_source_pids):
    contexts = build_inner_contexts(hb, caches, outer_target, outer_source_pids)
    score_rows = []
    audit_rows = []
    scores_A = {canonical_cfg(cfg): [] for cfg in A_SELECTION_GRID}
    for ctx in contexts:
        base_audit = hb.FitAudit(ctx['train_pids'], ctx['forbidden'], f"outer={outer_target}/inner={ctx['fold']}/BASE_for_A")
        base = fit_coherent_base(hb, ctx['train'], base_audit, stage_prefix='INNER_BASE_FOR_A')
        audit_rows.extend(base_audit.rows)
        for ci, cfg in enumerate(A_SELECTION_GRID):
            audit = hb.FitAudit(ctx['train_pids'], ctx['forbidden'], f"outer={outer_target}/inner={ctx['fold']}/A={ci}")
            model = fit_coherent_A(hb, ctx['train'], cfg, audit, stage_prefix='INNER_A')
            audit_rows.extend(audit.rows)
            loss = mean_e2e_loss(hb, ctx['validation_by_pid'], lambda x, b=base, m=model: predict_row_A(hb, b, m, x))
            scores_A[canonical_cfg(cfg)].append(loss)
            score_rows.append({'outer_target': outer_target, 'inner_fold': ctx['fold'], 'row': ROW_A, 'selection_stage': 'A_ONLY', 'inherited_cfg': 'FIXED_BASE', 'candidate_cfg': canonical_cfg(cfg), 'local_cfg': '', 'loss': loss, 'inner_validation_pids': '|'.join(ctx['held_pids'])})
    selected_A, loss_A = choose_configuration(scores_A, A_SELECTION_GRID)
    b_candidates = [{**selected_A, 'change_frac': float(frac)} for frac in B_CHANGE_FRAC_GRID]
    scores_B = {canonical_cfg(cfg): [] for cfg in b_candidates}
    for ctx in contexts:
        base_audit = hb.FitAudit(ctx['train_pids'], ctx['forbidden'], f"outer={outer_target}/inner={ctx['fold']}/BASE_for_B")
        base = fit_coherent_base(hb, ctx['train'], base_audit, stage_prefix='INNER_BASE_FOR_B')
        audit_rows.extend(base_audit.rows)
        for ci, cfg in enumerate(b_candidates):
            audit = hb.FitAudit(ctx['train_pids'], ctx['forbidden'], f"outer={outer_target}/inner={ctx['fold']}/B={ci}")
            model = fit_coherent_AB(hb, ctx['train'], cfg, audit, stage_prefix='INNER_AB')
            audit_rows.extend(audit.rows)
            loss = mean_e2e_loss(hb, ctx['validation_by_pid'], lambda x, b=base, m=model: predict_row_AB(hb, b, m, x))
            scores_B[canonical_cfg(cfg)].append(loss)
            score_rows.append({'outer_target': outer_target, 'inner_fold': ctx['fold'], 'row': ROW_AB, 'selection_stage': 'B_ONLY', 'inherited_cfg': canonical_cfg(selected_A), 'candidate_cfg': canonical_cfg(cfg), 'local_cfg': '', 'loss': loss, 'inner_validation_pids': '|'.join(ctx['held_pids'])})
    selected_AB, loss_AB = choose_configuration(scores_B, b_candidates)
    for key in ('state_C', 'timing_alpha'):
        if not np.isclose(float(selected_AB[key]), float(selected_A[key])):
            raise RuntimeError(f'B inheritance check failed for {key}.')
    c_candidates = [{**selected_AB, 'response_threshold': float(threshold)} for threshold in C_RESPONSE_THRESHOLD_GRID]
    scores_C = {canonical_cfg(cfg): [] for cfg in c_candidates}
    for ctx in contexts:
        for ci, cfg in enumerate(c_candidates):
            audit = hb.FitAudit(ctx['train_pids'], ctx['forbidden'], f"outer={outer_target}/inner={ctx['fold']}/C={ci}")
            model = fit_coherent_ABC(hb, ctx['train'], cfg, audit, stage_prefix='INNER_ABC')
            audit_rows.extend(audit.rows)
            loss = mean_e2e_loss(hb, ctx['validation_by_pid'], lambda x, m=model: m.predict(x))
            scores_C[canonical_cfg(cfg)].append(loss)
            score_rows.append({'outer_target': outer_target, 'inner_fold': ctx['fold'], 'row': ROW_ABC, 'selection_stage': 'C_ONLY', 'inherited_cfg': canonical_cfg(selected_AB), 'candidate_cfg': canonical_cfg(cfg), 'local_cfg': '', 'loss': loss, 'inner_validation_pids': '|'.join(ctx['held_pids'])})
    selected_ABC, loss_ABC = choose_configuration(scores_C, c_candidates)
    for key in ('state_C', 'timing_alpha', 'change_frac'):
        if not np.isclose(float(selected_ABC[key]), float(selected_AB[key])):
            raise RuntimeError(f'C inheritance check failed for {key}.')
    scores_LBR = {canonical_cfg(cfg): [] for cfg in LBR_SELECTION_GRID}
    for ctx in contexts:
        for li, lcfg in enumerate(LBR_SELECTION_GRID):
            audit = hb.FitAudit(ctx['train_pids'], ctx['forbidden'], f"outer={outer_target}/inner={ctx['fold']}/LBR={li}")
            model = fit_coherent_full(hb, ctx['train'], selected_ABC, lcfg, audit, outer_forbidden=ctx['forbidden'], context=f"outer={outer_target}/inner={ctx['fold']}/LBR={li}")
            audit_rows.extend(audit.rows)
            loss = mean_e2e_loss(hb, ctx['validation_by_pid'], lambda x, m=model: m.predict(x))
            scores_LBR[canonical_cfg(lcfg)].append(loss)
            score_rows.append({'outer_target': outer_target, 'inner_fold': ctx['fold'], 'row': ROW_FULL, 'selection_stage': 'LBR_ONLY', 'inherited_cfg': canonical_cfg(selected_ABC), 'candidate_cfg': canonical_cfg(selected_ABC), 'local_cfg': canonical_cfg(lcfg), 'loss': loss, 'inner_validation_pids': '|'.join(ctx['held_pids'])})
    selected_LBR, loss_FULL = choose_configuration(scores_LBR, LBR_SELECTION_GRID)
    selected = {ROW_A: {'global': dict(selected_A), 'local': None, 'inner_loss': float(loss_A), 'selection_stage': 'A_ONLY'}, ROW_AB: {'global': dict(selected_AB), 'local': None, 'inner_loss': float(loss_AB), 'selection_stage': 'B_ONLY_INHERITS_A'}, ROW_ABC: {'global': dict(selected_ABC), 'local': None, 'inner_loss': float(loss_ABC), 'selection_stage': 'C_ONLY_INHERITS_AB'}, ROW_FULL: {'global': dict(selected_ABC), 'local': dict(selected_LBR), 'inner_loss': float(loss_FULL), 'selection_stage': 'LBR_ONLY_INHERITS_ABC'}}
    if canonical_cfg(selected[ROW_FULL]['global']) != canonical_cfg(selected[ROW_ABC]['global']):
        raise RuntimeError('LBR selection changed the selected A+B+C parent.')
    return (selected, pd.DataFrame(score_rows), pd.DataFrame(audit_rows))

def run_hierarchical_selection_checks(source, hb):
    run_model_checks(source, hb)
    assert len(A_SELECTION_GRID) == 5
    assert B_CHANGE_FRAC_GRID == (0.2, 0.3, 0.4)
    assert C_RESPONSE_THRESHOLD_GRID == (0.4, 0.5, 0.6)
    assert len(LBR_SELECTION_GRID) == 7
    selected_A = dict(state_C=0.01, timing_alpha=10.0)
    selected_AB = {**selected_A, 'change_frac': 0.3}
    selected_ABC = {**selected_AB, 'response_threshold': 0.5}
    for key in ('state_C', 'timing_alpha'):
        assert selected_AB[key] == selected_A[key]
    for key in ('state_C', 'timing_alpha', 'change_frac'):
        assert selected_ABC[key] == selected_AB[key]
    print('Hierarchical selection checks: PASS')


In [ ]:
# Cell 10: Run VP001–VP026

from __future__ import annotations

def outer_all_rows(hb, caches, target, selected):
    source_pids = [p for p in ALL_PIDS if p != target]
    train = training_rows(hb, caches, source_pids)
    truth = caches[target]['test']
    x = hb.predictor_view(truth)
    rows = []
    audits = []
    audit = hb.FitAudit(source_pids, [target], f'coherent/outer={target}/BASE')
    base = fit_coherent_base(hb, train, audit, 'OUTER_BASE')
    pred_base = base.predict(x)
    rows.append(coherent_frame(target, ROW_BASE, truth, pred_base, selected_global='FIXED_CONVENTIONAL_BASE'))
    audits.extend(audit.rows)
    acfg = selected[ROW_A]['global']
    audit = hb.FitAudit(source_pids, [target], f'coherent/outer={target}/A')
    a_model = fit_coherent_A(hb, train, acfg, audit, 'OUTER_A')
    pred_a = predict_row_A(hb, base, a_model, x)
    rows.append(coherent_frame(target, ROW_A, truth, pred_a, selected_global=canonical_cfg(acfg), inner_loss=selected[ROW_A]['inner_loss']))
    audits.extend(audit.rows)
    abcfg = selected[ROW_AB]['global']
    audit = hb.FitAudit(source_pids, [target], f'coherent/outer={target}/AB')
    ab_model = fit_coherent_AB(hb, train, abcfg, audit, 'OUTER_AB')
    pred_ab = predict_row_AB(hb, base, ab_model, x)
    rows.append(coherent_frame(target, ROW_AB, truth, pred_ab, selected_global=canonical_cfg(abcfg), inner_loss=selected[ROW_AB]['inner_loss']))
    audits.extend(audit.rows)
    abccfg = selected[ROW_ABC]['global']
    audit = hb.FitAudit(source_pids, [target], f'coherent/outer={target}/ABC')
    abc_model = fit_coherent_ABC(hb, train, abccfg, audit, 'OUTER_ABC')
    pred_abc = abc_model.predict(x)
    rows.append(coherent_frame(target, ROW_ABC, truth, pred_abc, selected_global=canonical_cfg(abccfg), inner_loss=selected[ROW_ABC]['inner_loss']))
    audits.extend(audit.rows)
    fg = selected[ROW_FULL]['global']
    fl = selected[ROW_FULL]['local']
    audit = hb.FitAudit(source_pids, [target], f'coherent/outer={target}/FULL')
    full_model = fit_coherent_full(hb, train, fg, fl, audit, outer_forbidden={target}, context=f'coherent/outer={target}/FULL')
    pred_full = full_model.predict(x)
    rows.append(coherent_frame(target, ROW_FULL, truth, pred_full, selected_global=canonical_cfg(fg), selected_local=canonical_cfg(fl), inner_loss=selected[ROW_FULL]['inner_loss']))
    audits.extend(audit.rows)
    return (pd.concat(rows, ignore_index=True), pd.DataFrame(audits))

def repeat_seed_directory(parent: Path, seed: int) -> Path:
    return parent / f'seed_{int(seed):02d}'

def repeat_completion_marker(parent: Path, seed: int) -> Path:
    return repeat_seed_directory(parent, seed) / 'audit' / 'SEED_COMPLETE.json'

def build_inheritance_check(selected_by_outer):
    rows = []
    for pid in ALL_PIDS:
        selected = selected_by_outer[pid]
        A = selected[ROW_A]['global']
        AB = selected[ROW_AB]['global']
        ABC = selected[ROW_ABC]['global']
        FULL = selected[ROW_FULL]['global']
        LBR = selected[ROW_FULL]['local']
        a_to_b = all((np.isclose(float(AB[key]), float(A[key])) for key in ('state_C', 'timing_alpha')))
        b_to_c = all((np.isclose(float(ABC[key]), float(AB[key])) for key in ('state_C', 'timing_alpha', 'change_frac')))
        c_to_lbr = canonical_cfg(ABC) == canonical_cfg(FULL)
        rows.append({'participant': pid, 'A_cfg': canonical_cfg(A), 'AB_cfg': canonical_cfg(AB), 'ABC_cfg': canonical_cfg(ABC), 'LBR_global_parent': canonical_cfg(FULL), 'LBR_cfg': canonical_cfg(LBR), 'A_to_B_inheritance_pass': bool(a_to_b), 'AB_to_C_inheritance_pass': bool(b_to_c), 'ABC_to_LBR_parent_lock_pass': bool(c_to_lbr), 'pass': bool(a_to_b and b_to_c and c_to_lbr)})
    out = pd.DataFrame(rows)
    if not out['pass'].all():
        raise RuntimeError('Hierarchical inheritance check failed:\n' + out[~out['pass']].to_string(index=False))
    return out

def selected_configuration_frame(selected_by_outer):
    rows = []
    for pid in ALL_PIDS:
        selected = selected_by_outer[pid]
        rows.append({'participant': pid, 'row': ROW_BASE, 'configuration': ROW_LABELS[ROW_BASE], 'global_cfg': 'FIXED_CONVENTIONAL_BASE', 'local_cfg': '', 'inner_selected_loss': np.nan, 'selection_stage': 'FIXED_BASE'})
        for row in (ROW_A, ROW_AB, ROW_ABC, ROW_FULL):
            item = selected[row]
            rows.append({'participant': pid, 'row': row, 'configuration': ROW_LABELS[row], 'global_cfg': canonical_cfg(item['global']), 'local_cfg': canonical_cfg(item['local']) if item.get('local') is not None else '', 'inner_selected_loss': float(item['inner_loss']), 'selection_stage': item.get('selection_stage', '')})
    return pd.DataFrame(rows)

def merge_repeat_results(hb, seed_out: Path, seed: int):
    ck = seed_out / 'checkpoints'
    required = []
    for pid in ALL_PIDS:
        bundle = [ck / f'{pid}_ROWS.csv.gz', ck / f'{pid}_SELECTED.json', ck / f'{pid}_INNER.csv.gz', ck / f'{pid}_AUDIT.csv.gz']
        if not all((p.is_file() for p in bundle)):
            return False
        required.append((pid, bundle))
    all_rows = pd.concat([pd.read_csv(bundle[0]) for _pid, bundle in required], ignore_index=True)
    all_inner = pd.concat([pd.read_csv(bundle[2]) for _pid, bundle in required], ignore_index=True)
    all_audit = pd.concat([pd.read_csv(bundle[3]) for _pid, bundle in required], ignore_index=True)
    selected_by_outer = {pid: json.loads(bundle[1].read_text(encoding='utf-8')) for pid, bundle in required}
    inheritance = build_inheritance_check(selected_by_outer)
    pm = coherent_participant_metrics(hb, all_rows)
    summary = coherent_summary(pm)
    stats = coherent_incremental_stats(pm)
    selected = selected_configuration_frame(selected_by_outer)
    for df in (all_rows, all_inner, all_audit, inheritance, pm, summary, stats, selected):
        if 'repeat_seed' not in df.columns:
            df.insert(0, 'repeat_seed', int(seed))
    atomic_csv(all_rows, seed_out / 'results' / 'HemoBoundary_Per_Trial.csv.gz')
    atomic_csv(pm, seed_out / 'results' / 'HemoBoundary_Participant_Metrics.csv')
    atomic_csv(summary, seed_out / 'tables' / 'Table1_Component_Ablation.csv')
    atomic_csv(stats, seed_out / 'tables' / 'Table2_Incremental_Statistics.csv')
    atomic_csv(selected, seed_out / 'tables' / 'Table3_Selected_Configurations.csv')
    atomic_csv(inheritance, seed_out / 'audit' / 'Hierarchical_Inheritance_Check.csv')
    atomic_csv(all_inner, seed_out / 'audit' / 'ALL_INNER_CANDIDATE_SCORES.csv.gz')
    atomic_csv(all_audit, seed_out / 'audit' / 'ALL_FIT_MEMBERSHIP_AUDIT.csv.gz')
    atomic_json({'analysis': ANALYSIS_NAME, 'repeat_seed': int(seed), 'complete': True, 'n_outer_participants': 26, 'outer_evaluation': 'participant LOSO', 'inner_selection': '3-fold participant-grouped CV', 'outer_target_used_in_tuning': False, 'fresh_seed_specific_bank': True, 'hierarchical_selection': True, 'A_tunes': ['state_C', 'timing_alpha'], 'B_inherits_A': True, 'B_tunes': ['change_frac'], 'C_inherits_AB': True, 'C_tunes': ['response_threshold'], 'LBR_inherits_ABC': True, 'LBR_tunes': ['radius', 'local_alpha', 'strength'], 'LBR_source_participant_OOF_anchors': True, 'strict_inheritance_audit_pass': bool(inheritance['pass'].all())}, seed_out / 'audit' / 'SEED_COMPLETE.json')
    print('\n' + '=' * 180)
    print(f'SEED {seed:02d} — hierarchical component analysis')
    print('=' * 180)
    print(summary.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
    print('\n' + '=' * 180)
    print(f'SEED {seed:02d} — INCREMENTAL PAIRED RESULTS')
    print('=' * 180)
    print(stats.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
    print(f'\nSEED {seed:02d}: HIERARCHICAL INHERITANCE PASS 26/26')
    return True

def run_one_repeat(source, hb, shin_root: Path, fmap, parent_out: Path, seed: int, args):
    global COHERENT_SEED
    COHERENT_SEED = int(seed)
    configure_repeat_seed(source, hb, int(seed))
    seed_out = repeat_seed_directory(parent_out, seed)
    complete_marker = repeat_completion_marker(parent_out, seed)
    if complete_marker.is_file() and (not args.force) and (args.max_participants <= 0):
        print(f'\nSEED {seed:02d}: Completion marker found — skipping.')
        return True
    if args.force and seed_out.exists():
        print(f'SEED {seed:02d}: --force removes ONLY {seed_out}')
        shutil.rmtree(seed_out)
    for sub in ('fresh_cache', 'checkpoints', 'results', 'tables', 'audit'):
        (seed_out / sub).mkdir(parents=True, exist_ok=True)
    print('\n' + '=' * 180)
    print(f'HEMOBOUNDARY — hierarchical ROBUSTNESS SEED {seed:02d}')
    print('=' * 180)
    print('Runner part                   :', RUN_PART_NAME)
    print('Assigned seeds                :', list(RUN_SEEDS))
    print('Current seed                  :', int(seed))
    print('Outer targets                 : ALL 26 participants')
    print('Outer evaluation              : participant LOSO')
    print('Inner selection               : 3-fold participant-grouped CV')
    print('A                             : tune state_C + timing_alpha')
    print('B                             : inherit A; tune change_frac only')
    print('C                             : inherit A+B; tune response_threshold only')
    print('LBR                           : inherit ABC; tune radius/alpha/strength only')
    print('LBR residual anchors          : source-participant OOF')
    print('Outer target used in tuning   : No')
    print('Fresh semi-synthetic bank       : YES')
    print('Seed-specific inner folds     : YES')
    print('Seed-specific LBR OOF folds   : YES')
    print('Seed-specific estimator state : YES')
    print('Historical predictions reused : No')
    print('Output                        :', seed_out)
    caches0 = source.prepare_all_caches(fmap, seed_out / 'fresh_cache', mode='B')
    caches = add_pid_weight(caches0)
    cache_audit = pd.DataFrame([{'repeat_seed': int(seed), 'participant': pid, 'path': str(seed_out / 'fresh_cache' / f'{pid}.npz'), 'sha256': file_sha256(seed_out / 'fresh_cache' / f'{pid}.npz'), 'train_rows': len(caches[pid]['train']['present']), 'test_rows': len(caches[pid]['test']['present'])} for pid in ALL_PIDS])
    atomic_csv(cache_audit, seed_out / 'audit' / 'FRESH_CACHE_AUDIT.csv')
    targets = list(ALL_PIDS)
    if args.max_participants > 0:
        targets = targets[:args.max_participants]
    for oi, target in enumerate(targets, 1):
        ck = seed_out / 'checkpoints'
        cp_rows = ck / f'{target}_ROWS.csv.gz'
        cp_sel = ck / f'{target}_SELECTED.json'
        cp_inner = ck / f'{target}_INNER.csv.gz'
        cp_audit = ck / f'{target}_AUDIT.csv.gz'
        if cp_rows.is_file() and cp_sel.is_file() and cp_inner.is_file() and cp_audit.is_file() and (not args.force):
            print(f'[seed {seed:02d}] [{oi:02d}/{len(targets):02d}] {target}: RESUME', flush=True)
            continue
        print(f'[seed {seed:02d}] [{oi:02d}/{len(targets):02d}] {target}: hierarchical nested selection', flush=True)
        source_pids = [p for p in ALL_PIDS if p != target]
        selected, inner_scores, inner_audit = select_coherent_configs(hb, caches, target, source_pids)
        print('    +A   :', selected[ROW_A])
        print('    +B   :', selected[ROW_AB])
        print('    +C   :', selected[ROW_ABC])
        print('    +LBR :', selected[ROW_FULL])
        rows, outer_audit = outer_all_rows(hb, caches, target, selected)
        rows.insert(0, 'repeat_seed', int(seed))
        inner_scores.insert(0, 'repeat_seed', int(seed))
        combined_audit = pd.concat([inner_audit, outer_audit], ignore_index=True)
        combined_audit.insert(0, 'repeat_seed', int(seed))
        atomic_csv(rows, cp_rows)
        atomic_csv(inner_scores, cp_inner)
        atomic_csv(combined_audit, cp_audit)
        atomic_json(selected, cp_sel)
        del (rows, inner_scores, inner_audit, outer_audit, combined_audit, selected)
        gc.collect()
    if args.max_participants > 0:
        print(f'\nSEED {seed:02d}: smoke/partial run complete ({len(targets)}/26 outer participants).')
        return False
    merged = merge_repeat_results(hb, seed_out, seed)
    if not merged:
        raise RuntimeError(f'SEED {seed:02d}: expected all 26 checkpoint bundles, but merge found an incomplete participant.')
    return True


In [ ]:
# Cell 11: Repeat seeds 1–10

from __future__ import annotations

def configure_repeat_seed(source, hb, seed: int) -> None:
    global ACTIVE_RUN_SEED
    ACTIVE_RUN_SEED = int(seed)
    if not hasattr(source, '_MULTISEED_BASE_STABLE_SEED'):
        source._MULTISEED_BASE_STABLE_SEED = source.stable_seed
        source._MULTISEED_BASE_RANDOM_SEED = int(source.RANDOM_SEED)
    if not hasattr(hb, '_MULTISEED_BASE_STABLE_SEED'):
        hb._MULTISEED_BASE_STABLE_SEED = hb.stable_seed
        hb._MULTISEED_BASE_RANDOM_SEED = int(hb.RANDOM_SEED)
    source_base = source._MULTISEED_BASE_STABLE_SEED
    hb_base = hb._MULTISEED_BASE_STABLE_SEED
    if int(seed) == 1:
        source.stable_seed = source_base
        hb.stable_seed = hb_base
        source.RANDOM_SEED = int(source._MULTISEED_BASE_RANDOM_SEED)
        hb.RANDOM_SEED = int(hb._MULTISEED_BASE_RANDOM_SEED)
    else:

        def source_seed(*parts, _seed=int(seed), _base=source_base):
            return _base('MULTISEED_REPEAT', _seed, *parts)

        def hb_seed(*parts, _seed=int(seed), _base=hb_base):
            return _base('MULTISEED_REPEAT', _seed, *parts)
        source.stable_seed = source_seed
        hb.stable_seed = hb_seed
        source.RANDOM_SEED = int(source._MULTISEED_BASE_RANDOM_SEED) + int(seed) - 1
        hb.RANDOM_SEED = int(hb._MULTISEED_BASE_RANDOM_SEED) + int(seed) - 1
    np.random.seed(base_stable_seed('numpy_legacy', int(seed)) % (2 ** 32 - 1))

def print_assigned_seed_summary(parent_out: Path):
    completed = [int(seed) for seed in RUN_SEEDS if repeat_completion_marker(parent_out, seed).is_file()]
    if not completed:
        return
    tables = []
    for seed in completed:
        path = repeat_seed_directory(parent_out, seed) / 'tables' / 'Table1_Component_Ablation.csv'
        if path.is_file():
            q = pd.read_csv(path)
            if 'repeat_seed' not in q.columns:
                q.insert(0, 'repeat_seed', seed)
            tables.append(q)
    if not tables:
        return
    all_table = pd.concat(tables, ignore_index=True)
    metrics = ('response_balanced_accuracy', 'onset_mae_sec', 'peak_mae_sec', 'offset_mae_sec', 'duration_mae_sec', 'selection_loss')
    rows = []
    for row in ROW_ORDER:
        q = all_table[all_table['row'].astype(str) == row]
        if len(q) == 0:
            continue
        r = {'row': row, 'configuration': ROW_LABELS[row], 'n_completed_assigned_seeds': int(q['repeat_seed'].nunique())}
        for metric in metrics:
            x = pd.to_numeric(q[metric], errors='coerce').to_numpy(float)
            r[metric + '_mean'] = float(np.nanmean(x))
            r[metric + '_sd'] = float(np.nanstd(x, ddof=1)) if len(x) > 1 else 0.0
        rows.append(r)
    summary = pd.DataFrame(rows)
    out = parent_out / 'PART_SUMMARIES'
    out.mkdir(parents=True, exist_ok=True)
    atomic_csv(all_table, out / f'{RUN_PART_NAME}_PER_SEED_TABLE1.csv')
    atomic_csv(summary, out / f'{RUN_PART_NAME}_MEAN_SD.csv')
    print('\n' + '=' * 180)
    print(f'{RUN_PART_NAME} — CURRENT ASSIGNED-SEED MEAN / SD')
    print('=' * 180)
    print(summary.to_string(index=False, float_format=lambda x: f'{x:.6f}'))

def parse_arguments():
    p = argparse.ArgumentParser()
    p.add_argument('--shin-root', default=None)
    p.add_argument('--output', default=None)
    p.add_argument('--max-participants', type=int, default=0, help='Smoke-test only the first N outer participants of each assigned seed.')
    p.add_argument('--force', action='store_true', help='Delete and rerun all ten unified seed folders.')
    p.add_argument('--merge-only', action='store_true', help='Do no training; merge/print seeds that already have all 26 participant checkpoints.')
    p.add_argument('--self-test-only', action='store_true')
    args, unknown = p.parse_known_args()
    if unknown:
        print('Ignoring notebook/runtime arguments:', unknown)
    return args


In [ ]:
# Cell 12: Merge results and print paper results

from __future__ import annotations

def run_main_analysis():
    if RUN_PART_NAME != EXPECTED_RUN_PART_NAME:
        raise RuntimeError(f'Runner identity mismatch: {RUN_PART_NAME} != {EXPECTED_RUN_PART_NAME}')
    if tuple(RUN_SEEDS) != tuple(EXPECTED_RUN_SEEDS):
        raise RuntimeError(f'Runner seed assignment mismatch: {RUN_SEEDS} != {EXPECTED_RUN_SEEDS}')
    if tuple(RUN_SEEDS) != tuple(range(1, 11)):
        raise RuntimeError(f'Expected exactly seeds 1..10, got {RUN_SEEDS}.')
    print(f'HARD SEED ASSIGNMENT CHECK: PASS — {RUN_PART_NAME} owns seeds {list(RUN_SEEDS)}')
    args = parse_arguments()
    with tempfile.TemporaryDirectory(prefix='hemoboundary_strict_multiseed_') as td:
        td = Path(td)
        source = SOURCE_NS
        hb = HB_NS
        configure_repeat_seed(source, hb, 1)
        global COHERENT_SEED
        COHERENT_SEED = 1
        run_hierarchical_selection_checks(source, hb)
        if args.self_test_only:
            return
        shin_root = find_root(args.shin_root)
        fmap = source.participant_files(shin_root)
        if sorted(fmap) != list(ALL_PIDS):
            raise RuntimeError(f'Expected exactly SHIN VP001..VP026 preprocessing files. Found: {sorted(fmap)}')
        parent_out = Path(args.output) if args.output else shin_root / MAIN_OUTPUT_DIRNAME
        parent_out.mkdir(parents=True, exist_ok=True)
        print('=' * 180)
        print('HEMOBOUNDARY — UNIFIED hierarchical 10-SEED × 26-PARTICIPANT LOSO')
        print('=' * 180)
        print('Runner part                  :', RUN_PART_NAME)
        print('Assigned seeds               :', list(RUN_SEEDS))
        print('Participants / seed          : 26')
        print('Fresh bank / seed            : YES')
        print('Architecture changed         : No')
        print('Hyperparameter grids changed : No')
        print('Hierarchical inheritance changed: No')
        print('Output root                  :', parent_out)
        if args.merge_only:
            for seed in RUN_SEEDS:
                seed_out = repeat_seed_directory(parent_out, seed)
                ok = merge_repeat_results(hb, seed_out, seed)
                print(f'Seed {seed:02d} merge:', 'Finished' if ok else 'Incomplete')
            print_assigned_seed_summary(parent_out)
            return
        if args.force:
            for seed in RUN_SEEDS:
                seed_out = repeat_seed_directory(parent_out, seed)
                if seed_out.exists():
                    print('--force removes assigned seed only:', seed_out)
                    shutil.rmtree(seed_out)
        for seed in RUN_SEEDS:
            run_one_repeat(source, hb, shin_root, fmap, parent_out, int(seed), args)
            gc.collect()
        print_assigned_seed_summary(parent_out)
        atomic_json({'analysis': ANALYSIS_NAME, 'runner_part': RUN_PART_NAME, 'assigned_seeds': list(RUN_SEEDS), 'participants_per_seed': 26, 'architecture': 'Base -> +A Temporal Boundary Estimation -> +B Recovery Estimation -> +C Response Detection -> +LBR Local Boundary Refinement', 'hierarchical_selection': True, 'A_tunes': ['state_C', 'timing_alpha'], 'B_inherits_A_and_tunes': ['change_frac'], 'C_inherits_AB_and_tunes': ['response_threshold'], 'LBR_inherits_ABC_and_tunes': ['radius', 'local_alpha', 'strength'], 'fresh_seed_specific_bank': True, 'outer_target_used_in_tuning': False, 'parallel_safe_non_overlapping_seed_folders': True}, parent_out / f'{RUN_PART_NAME}_RUNNER_MANIFEST.json')
        print('\n' + '=' * 180)
        print(f'{RUN_PART_NAME}: FINISHED')
        print('=' * 180)

run_main_analysis()
